# Bias Analysis - QD vs. Decision Trees
#### Code by Catalina Jaramillo and M Charity
---

### Experiments
[ ] - Evolving neural network weights

[ ] - Evolving decision trees

### Datasets



### Features
- gender + age + race --> hiring prediction

### Results summary


### Other Notes
 

---- 

### CODE SETUP

In [1]:
# imports

import math
import random
import itertools
import inspect
import time
import json

from tqdm import tqdm

# data proc
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# pyribs
from ribs.archives import GridArchive
from ribs.emitters import GaussianEmitter, EvolutionStrategyEmitter
from ribs.schedulers import Scheduler
import matplotlib.pyplot as plt
from ribs.visualize import grid_archive_heatmap     # be sure to install shapely for this to work


In [2]:
# import external classes and files
from EvoDecisionTree import EvoDecTree, EvoNode
from EvoNeuralNet import EvoNN

### UTILITY FUNCTIONS

In [3]:
# Combine lambdas for filtering rows in a DataFrame
def combine_lambdas(keys,l_dict):
    return lambda row: all(l_dict[k](row) for k in keys)

def get_combo_data(n,c,data,l_dict):
    ''' Returns a new dataset based on the given features 
        data is the original dataset
        c is the abbreviated combo
        l_dict is a dictionary of lambda functions to be used for filtering based on the features pairings
            - ex. ['f','p','y'] will filter for rows'''
    
    match = combine_lambdas(c,l_dict)
    m_data = data[data.apply(match,axis=1)]
    if m_data.shape[0] > 0:
        return m_data.sample(n)
    else:
        return pd.DataFrame()

# dataset sample creator
def unbiased_dataset(n,data,feats,l_dict):
    ''' Creates an unbiased dataset with all combinations of features the same size '''

    # Generate all combinations
    combos = list(itertools.product(*feats))
    combos = [list(comb) for comb in combos]

    # sample based on every combination
    new_data = pd.DataFrame()
    for c in combos:
        x = get_combo_data(n,c,data,l_dict)
        if x.shape[0] > 0:
            new_data = pd.concat([new_data, x])

    return new_data

def bias_n(n_bias, combo):
    ''' Returns the n value for the occurrence of a bias value '''
    for k,v in n_bias.items():
        if k == 'default':
            continue
        if all([i in combo for i in k]):        # has all the bias values
            return v

    return n_bias['default']

def bias_dataset(n_bias,data,feats,l_dict):
    ''' Creates a biased dataset with all combinations of features the same size for particular feature values '''

    # Generate all combinations
    combos = list(itertools.product(*feats))
    combos = [list(comb) for comb in combos]

    # sample based on every combination
    new_data = pd.DataFrame()
    for c in combos:
        # get the n value based on the bias in the dataset combo
        n = bias_n(n_bias,c)

        # get the matching data
        x = get_combo_data(n,c,data,l_dict)
        if x.shape[0] > 0:
            new_data = pd.concat([new_data, x])

    return new_data


In [4]:
def arx_vis(arx, x_prob, y_prob, a, b, scaled=False):
    plt.figure(figsize=(a, b))
    if scaled:
        grid_archive_heatmap(arx, vmin=0, vmax=1)
    else:
        grid_archive_heatmap(arx)
    plt.title("Accuracy")
    plt.xlabel(x_prob)
    plt.ylabel(y_prob)
    plt.show()

--- 
### TEST CODE

In [5]:
'''
mini_data = pd.DataFrame({'A': [6, 1, 0], 'B': [1, 1, 0], 'C': [-2, 0, 2]})
f = [['A', 'a'], ['B', 'b'], ['C', 'c']]
d = {
    'a': lambda x: x['A'] > 0,
    'b': lambda x: x['B'] == 1,
    'c': lambda x: x['C'] <= 0,
    'A': lambda x: x['A'] <= 0,
    'B': lambda x: x['B'] == 0,
    'C': lambda x: x['C'] > 0
}

print(unbiased_dataset(1, mini_data, f, d))
#print(mini_data)
'''

"\nmini_data = pd.DataFrame({'A': [6, 1, 0], 'B': [1, 1, 0], 'C': [-2, 0, 2]})\nf = [['A', 'a'], ['B', 'b'], ['C', 'c']]\nd = {\n    'a': lambda x: x['A'] > 0,\n    'b': lambda x: x['B'] == 1,\n    'c': lambda x: x['C'] <= 0,\n    'A': lambda x: x['A'] <= 0,\n    'B': lambda x: x['B'] == 0,\n    'C': lambda x: x['C'] > 0\n}\n\nprint(unbiased_dataset(1, mini_data, f, d))\n#print(mini_data)\n"

In [6]:
thresh = 34
COL_KEY = {
    'f': lambda x: x['female'] == 1,
    'm': lambda x: x['male'] == 1,
    'p': lambda x: x['is_promoted'] == 1,
    'n': lambda x: x['is_promoted'] == 0,
    'o': lambda x: x['age'] >= thresh,
    'y': lambda x: x['age'] < thresh
}
FEAT_SET = [['f', 'm'], ['p', 'n'], ['o', 'y']]


In [7]:
MAKE_BIAS_TEST = False

# make unbiased data
if MAKE_BIAS_TEST:
    # test dataset from the original paper
    test_dataset = pd.read_csv("./Classif-Bias-Tradeoffs-QD-main/promotion/train2.csv")
    test_dataset.drop('education', axis=1, inplace=True)

    unbias_data = unbiased_dataset(716, test_dataset, FEAT_SET, COL_KEY)
    print(unbias_data.shape)

In [8]:
# make biased data
if MAKE_BIAS_TEST:
    promo_bias = bias_dataset({'default': 716, 'n': 380}, test_dataset, FEAT_SET, COL_KEY)
    male_promo_bias = bias_dataset({'default': 716, 'pf': 380, 'pm': 1476}, test_dataset, FEAT_SET, COL_KEY)
    old_man_young_women_bias = bias_dataset({'default': 376, 'om': 376, 'yf': 376}, test_dataset, FEAT_SET, COL_KEY)
    old_man_bias = bias_dataset({'default': 376, 'f':564, 'om': 751}, test_dataset, FEAT_SET, COL_KEY)

---
### QD EXPERIMENT: Evolving NN Weights w/ CMA-ME

Rewriting Pyribs was a pain in the ASS :( why would anyone change the framework definition?!

In [9]:
def getArxSubInd(arx, xr=(12,17), yr=(12,17), dims=(30,30)):
    ''' Returns a subset of the archive based on the x and y indices '''
    i = list(range(dims[0]*dims[1]))
    i = np.array(i).reshape(dims)
    #print(i)

    # get only the indices in between the xr and yr range
    j = i[xr[0]:xr[1], yr[0]:yr[1]]
    return j.flatten().tolist()
    

In [10]:
a = 64
b = 32
c = 1

def batchFitNN(zs, X, y, X_set):
    ''' Batch generation of vectors for use in the CMA-ME experiment with NNs'''
    global a, b, c

    fs = []
    cs = []

    for z in zs:
        nn = EvoNN(z)
        nn.set_params(a=a, b=b, c=c)
        nn.set_weights(z)

        f,c = nn.run(X, y, X_set)
        fs.append(f)
        cs.append(c)
    return np.array(fs), np.array(cs)

def nnME(X,y, comparisons,iterations=100000):
    ''' CMA-ME experiment using vector-based NNs model'''
    global a, b, c

    # print(X.shape)
    # print(y.shape)

    rang = [[0, 2], [0, 2],[0,2]]
    dims = [30,30,30]
    s = ((X.shape[1]+1)*a)+((a+1)*b)+((b+1)*1)

    archive = GridArchive(solution_dim=s, ranges=rang, dims=dims)
    emitter1 = GaussianEmitter(archive, sigma=0.1, batch_size=32,
                              x0 = np.random.normal(scale=1, size=s))
    emitter2 = EvolutionStrategyEmitter(archive, sigma0=0.1, batch_size=32,
                              x0 = np.random.normal(scale=1, size=s))
    emitters = [emitter1, emitter2]
    opt = Scheduler(archive, emitters)       # optimizer was renamed to scheduler for some reason
    #opt.set_verbosity(1)

    # create case sets (always in pairs)
    X_set = []
    for c in comparisons:
        if "GENDER" in c:
            male = X[X['gender']==1]
            female = X[X['gender']==0]
            X_set.append(male)
            X_set.append(female)
        
        elif "AGE" in c:
            old = X[X['age_binary']==0]
            young = X[X['age_binary']==1]
            X_set.append(old)
            X_set.append(young)

        elif "RACE" in c:
            white = X[X['race']==1]
            other = X[X['race']==0]
            X_set.append(white)
            X_set.append(other)

    start_time = time.time()
    solutions = []
    with tqdm(total=iterations) as pbar:
        for itr in range(iterations):
            pbar.set_description(f"Iteration {itr} - {len(archive)} / {archive.cells} elites")
            sols = opt.ask()
        
            # Reshape and normalize the image and pass it through the network.
            zs = sols
            objs, bcs = batchFitNN(zs, X, y, X_set)
            
            # print(bcs)
            # print(f"objs: {objs} | bcs: {bcs}")
            # if itr%20 == 0: print (f"objs: {objs[0]} | bcs: {bcs[0]}")
            
            opt.tell(objs, bcs)
            #print(str(len(archive.as_pandas()['index_0'].values)))
        
            if itr % 1000 == 0:
                print(f"Iteration {itr} complete after {(time.time() - start_time)/60} m")
                print(str(len(archive.data(return_type='pandas')['measures_0'].values)))
                solutions.append(str(len(archive.data(return_type='pandas')['measures_0'].values)))
                if len(solutions)>50 and solutions[-1] == solutions[-50]:
                # if len(solutions)>10 and solutions[-1] == solutions[-10]:
                    print('stagnant')
                    break


            if itr % 1000 == 0:
                arx_vis(archive, comparisons[0], comparisons[1], 4,3)

            pbar.update(1)


    # export
    arx = archive.data(return_type='pandas')
    arx.to_pickle('nn_exp.pkl')

    # for models in the square (fair zone)
    square_i = getArxSubInd(arx, xr=(12,17), yr=(12,17), dims=(30,30))
    square = arx.loc[arx['index'].isin(square_i)]
        
    best_model_unbiased_index = square['objective'].idxmax() if len(square) > 0 else None
    best_model_index = arx['objective'].idxmax()
    
    best_model_weights = arx.loc[best_model_index].iloc[5:].values if best_model_index is not None else None
    best_model_unbiased_weights = arx.loc[best_model_unbiased_index].iloc[5:].values if best_model_unbiased_index is not None else None
    
    return best_model_weights, best_model_unbiased_weights, square, arx, best_model_index, best_model_unbiased_index


---
### DT EXPERIMENT: Evolving Decision Trees w/ CMA-ME

In [11]:
def batchFitDT(zs, X, y, X_set):
    ''' Batch generation of vectors for use in the CMA-ME experiment with DTs'''
    X_np = X.to_numpy()
    y_np = y.to_numpy().flatten()
    
    fs = []
    cs = []
    for z in zs:
        dt = EvoDecTree(X_np,y_np)
        #print(z)
        dt.build_tree_from_vec(z)

        #print(dt)      # show tree

        f,c = dt.run(X_np, y_np, X_set)
        fs.append(f)
        cs.append(c)

    return np.array(fs), np.array(cs)

def dtME(X,y, comparisons, iterations=100000):
    ''' CMA-ME experiment using vector-based Decision Tree model'''
    rang = [[0, 2], [0, 2]]
    dims = [30,30]
    s = 2**14

    archive = GridArchive(solution_dim=s, ranges=rang, dims=dims)
    emitter1 = GaussianEmitter(archive, sigma=0.1, batch_size=32,
                              x0 = np.random.randint(low=1,high=1000000, size=s))
    emitter2 = EvolutionStrategyEmitter(archive, sigma0=0.1, batch_size=32,
                              x0 = np.random.randint(low=1,high=1000000, size=s))
    emitters = [emitter1, emitter2]
    opt = Scheduler(archive, emitters)
    #opt.set_verbosity(1)

    # create case sets (always in pairs)
    X_set = []
    for c in comparisons:
        if "GENDER" in c:
            male = X[X['gender']==1]
            female = X[X['gender']==0]
            X_set.append(male.to_numpy())
            X_set.append(female.to_numpy())
        
        elif "AGE" in c:
            old = X[X['age_binary']==0]
            young = X[X['age_binary']==1]
            X_set.append(old.to_numpy())
            X_set.append(young.to_numpy())

        elif "RACE" in c:
            white = X[X['race']==1]
            other = X[X['race']==0]
            X_set.append(white.to_numpy())
            X_set.append(other.to_numpy())


    start_time = time.time()
    solutions = []
    with tqdm(total=iterations) as pbar:
        for itr in range(iterations):
            pbar.set_description(f"Iteration {itr} - {len(archive)} / {archive.cells} elites")
            sols = opt.ask()
        
            # Reshape and normalize the image and pass it through the network.
            zs = sols
            zs = [[int(i) for i in z] for z in zs]      # convert to int
            objs, bcs = batchFitDT(zs, X, y, X_set)
            
            # print(bcs)
            # print(f"objs: {objs} | bcs: {bcs}")
            # if itr%20 == 0: print (f"objs: {objs[0]} | bcs: {bcs[0]}")
            
            opt.tell(objs, bcs)
            #print(str(len(archive.as_pandas()['index_0'].values)))
        
            if itr % 1000 == 0:
                print(f"Iteration {itr} complete after {(time.time() - start_time)/60} m")
                print(str(len(archive.data(return_type='pandas')['measures_0'].values)))
                solutions.append(str(len(archive.data(return_type='pandas')['measures_0'].values)))
                if len(solutions)>50 and solutions[-1] == solutions[-50]:
                # if len(solutions)>10 and solutions[-1] == solutions[-10]:
                    print('stagnant')
                    break

            if itr % 1000 == 0:
                arx_vis(archive, comparisons[0], comparisons[1], 4,3)

            pbar.update(1)

    # export
    arx = archive.data(return_type='pandas')
    arx.to_pickle('nn_exp.pkl')

    # for models in the square (fair zone)
    square_i = getArxSubInd(arx, xr=(12,17), yr=(12,17), dims=(30,30))
    square = arx.loc[arx['index'].isin(square_i)]
        
    best_model_unbiased_index = square['objective'].idxmax() if not square.empty else None
    best_model_index = arx['objective'].idxmax()
    
    best_model_weights = arx.loc[best_model_index].iloc[5:].values if best_model_index is not None else None
    best_model_unbiased_weights = arx.loc[best_model_unbiased_index].iloc[5:].values if best_model_unbiased_index is not None else None
    
    return best_model_weights, best_model_unbiased_weights, square, arx, best_model_index, best_model_unbiased_index


### Run Experiments

In [12]:
EXPERIMENT = "BOTH"
EXP_RANGE = [1,2,3]
EXP_ITERATIONS = 100000

In [13]:
# IMPORT REAL DATASETS
### the adult, german, compass, and law datasets ###

# load individual dataset
synthetic = {}

comp_set = {
    'synthetic':['[GENDER] Male | Female','[RACE] Other | White','[AGE] <50 | 50+'], 
}



synthetic[1] = []
for p in ['X','y']:
    synthetic[1].append(pd.read_csv(f"./synthetic_hiring_bias_dataset_{p}.csv"))


In [14]:
# create directories if not there yet
import os
if not os.path.exists("./archive"):
    os.makedirs("./archive")
if not os.path.exists("./weights"):
    os.makedirs("./weights")
if not os.path.exists("./results"):
    os.makedirs("./results")

for d in ['NN', 'DT']:
    if not os.path.exists(f"./archive/{d}"):
        os.makedirs(f"./archive/{d}")
    if not os.path.exists(f"./weights/{d}"):
        os.makedirs(f"./weights/{d}")
    if not os.path.exists(f"./results/{d}"):
        os.makedirs(f"./results/{d}")

In [15]:
# experiment exporter

def export_weights(weights, test, name):
    ''' Export the weights to a file (csv) '''
    with open(f"./weights/{test}/{name}.csv", "w") as f:
        pd.DataFrame(weights).to_csv(f, index=False, header=False)


def export_results(results, test, name):
    ''' Export the results to a file '''
    with open(f"./results/{test}/{name}.txt", "w") as f:
        for result in results:
            f.write(f"{result}\n")

In [16]:
# NEURAL NETWORK EXPERIMENT
if EXPERIMENT == "NN" or EXPERIMENT == "BOTH":
    # NN experiment
    solutions = []
    for d in ['synthetic']:
        for i in EXP_RANGE:
            print("="*30)
            print(f"\nRunning NN experiment {i} on {d} dataset\n")
            print("="*30)
            X = synthetic[i][0]
            y = synthetic[i][1]

            # split the data into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

            scaler=MinMaxScaler()
            X_train= pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
            X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

            comps = comp_set[d]

            best_model_weights, best_model_unbiased_weights, square, arx, best_model_index, best_model_unbiased_index = nnME(X,y, comps, iterations=EXP_ITERATIONS)
            print(f"Best model weights: {str(best_model_weights)}")
            print(f"Best unbiased model weights: {str(best_model_unbiased_weights)}")
            print(f"Best model index: {best_model_index}")
            print(f"Best unbiased model index: {best_model_unbiased_index}")

            # save the model
            date_time = time.strftime("%Y-%m-%d")
            export_weights(best_model_weights, "NN", f"{d}_{i}-[{date_time}]")

            # export the results
            res = []
            res.append(f"== DATA: {d.upper()} {i} ==")
            res.append(f"Iterations: {EXP_ITERATIONS}")
            res.append(f"Best model weights: {str(best_model_weights)}")
            res.append(f"Best unbiased model weights: {str(best_model_unbiased_weights)}")
            res.append(f"Best model index: {best_model_index}")
            res.append(f"Best unbiased model index: {best_model_unbiased_index}")
            export_results(res, "NN", f"{d}_{i}-[{date_time}]")

            # save the archive
            arx.to_pickle(f"./archive/NN/{d}_{i}-[{date_time}].pkl")




Running NN experiment 1 on synthetic dataset



Iteration 0 - 0 / 27000 elites:   0%|          | 0/100000 [00:00<?, ?it/s]

Iteration 0 - 0 / 27000 elites:   0%|          | 0/100000 [00:00<?, ?it/s]


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# Decision Tree experiment
if EXPERIMENT == "DT" or EXPERIMENT == "BOTH":

    # DT experiment
    solutions = []
    for d in ['synthetic']:
        for i in EXP_RANGE:
            print("="*30)
            print(f"\nRunning DT experiment {i} on {d} dataset\n")
            print("="*30)

            X = eval(d)[i][0]
            y = eval(d)[i][1]

            comps = comp_set[d]
            
            best_model_vec, best_model_unbiased_vec, square, arx, best_model_index, best_model_unbiased_index = dtME(X,y, comps, iterations=EXP_ITERATIONS)
            print(f"Best model vector: {str(best_model_vec)}")
            print(f"Best unbiased model vector: {str(best_model_unbiased_vec)}")
            print(f"Best model index: {best_model_index}")
            print(f"Best unbiased model index: {best_model_unbiased_index}")


            # save the model
            date_time = time.strftime("%Y-%m-%d")
            export_weights(best_model_weights, "DT", f"{d}_{i}-[{date_time}]")
            with open(f"./weights/DT/{d}_{i}-[{date_time}].csv", "w") as f:
                f.write(f"{best_model_vec}\n")

            # export the results
            res = []
            res.append(f"== DATA: {d.upper()} {i} ==")
            res.append(f"Iterations: {EXP_ITERATIONS}")
            res.append(f"Best model vector: {str(best_model_vec)}")
            res.append(f"Best unbiased model vector: {str(best_model_unbiased_vec)}")
            res.append(f"Best model index: {best_model_index}")
            res.append(f"Best unbiased model index: {best_model_unbiased_index}")
            export_results(res, "DT", f"{d}_{i}-[{date_time}]")

            # save the archive
            arx.to_pickle(f"./archive/DT/{d}_{i}-[{date_time}].pkl")

### Debug Tests - Fixing pyribs sub archive

In [ ]:
import pandas as pd
import numpy as np
arx_debug = pd.read_pickle('nn_exp.pkl')
print(type(arx_debug))

In [ ]:
def getArxSubInd(arx, xr, yr, dims=[30,30]):
    ''' Returns a subset of the archive based on the x and y indices '''
    i = list(range(dims[0]*dims[1]))
    i = np.array(i).reshape(dims)
    #print(i)

    # get only the indices in between the xr and yr range
    j = i[xr[0]:xr[1], yr[0]:yr[1]]
    return j.flatten().tolist()
    

In [ ]:
sub_arx_i = getArxSubInd(arx_debug, [12,17], [12,17], [30,30])
sub_arc = arx_debug.loc[arx_debug['index'].isin(sub_arx_i)]
print(sub_arc.shape)
print(sub_arc['objective'].idxmax())
#print(arx_debug.loc[12:18, 12:18].shape)
#print(arx_debug.iloc[12:18, 12:18].get_field('objective').values)

In [ ]:
#square = arx_debug[(arx_debug.index_0>=12) & (arx_debug.index_1>=12) & (arx_debug.index_0<=17) & (arx_debug.index_1<=17)]
#print(arx_debug.index_0)
print(square.shape)

In [ ]:

best_model_unbiased_index = square['objective'].idxmax() if len(square) > 0 else None
best_model_index = arx_debug['objective'].idxmax()

best_model_weights = arx_debug.loc[best_model_index].iloc[5:].values if best_model_index is not None else None
best_model_unbiased_weights = arx_debug.loc[best_model_unbiased_index].iloc[5:].values if best_model_unbiased_index is not None else None


print(f"Best model unbiased index: {best_model_unbiased_index}")
print(f"Best model index: {best_model_index}")
print(f"Best model unbiased weights: {best_model_unbiased_weights}")
print(f"Best model weights: {best_model_weights}")